# Formula 1 Performance Analytics: 1950-2026

A research-grade notebook for driver, constructor, circuit, qualifying, and race outcome analysis.

This notebook builds a full Formula 1 sports analytics workflow across circuits, drivers, constructors, races, results, qualifying, driver standings, and constructor standings. It validates schemas, creates robust joins, engineers era-aware performance metrics, visualizes historical patterns, builds predictive models, clusters driver archetypes, and finishes with a Monte Carlo race simulation engine.

Research principles:

1. Normalize across eras because points systems, reliability, field sizes, and calendar lengths changed dramatically.
2. Avoid predictive leakage by using pre-race and shifted historical features in machine learning.
3. Prefer interpretable scores so every ranking can be audited and adjusted.
4. Combine visual storytelling with reproducible code.


## 1. Environment Setup

The notebook uses pandas, numpy, matplotlib, seaborn, scikit-learn, and XGBoost when available. If XGBoost is unavailable, the rest of the notebook remains executable and reports the skipped models.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBRegressor, XGBClassifier
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    XGBOOST_IMPORT_ERROR = repr(exc)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(context='notebook', style='whitegrid', palette='deep', rc={'figure.figsize': (12, 6), 'axes.spines.top': False, 'axes.spines.right': False, 'axes.titleweight': 'bold', 'axes.titlesize': 15, 'axes.labelsize': 11, 'legend.frameon': False})
plt.rcParams['figure.dpi'] = 130
pd.set_option('display.max_columns', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

DATA_DIR = Path(r'W:/archive')
if not DATA_DIR.exists():
    DATA_DIR = Path('.')

print(f'Using data directory: {DATA_DIR.resolve()}')
print('XGBoost available:', HAS_XGBOOST)
if not HAS_XGBOOST:
    print('XGBoost import issue:', XGBOOST_IMPORT_ERROR)


## 2. Load Data and Validate Schemas

The archive uses season and round as the race key, plus driver_id, constructor_id, and circuit_id as entity keys. Qualifying begins in 1994, so missing qualifying data before then is structural rather than an error.


In [ ]:
FILES = {
    'circuits': 'f1_circuits.csv',
    'drivers': 'f1_drivers.csv',
    'constructors': 'f1_constructors.csv',
    'races': 'f1_races.csv',
    'results': 'f1_results.csv',
    'qualifying': 'f1_qualifying.csv',
    'driver_standings': 'f1_driver_standings.csv',
    'constructor_standings': 'f1_constructor_standings.csv',
}
required_columns = {
    'circuits': {'circuit_id', 'circuit_name', 'country', 'lat', 'lng'},
    'drivers': {'driver_id', 'given_name', 'family_name', 'dob', 'nationality'},
    'constructors': {'constructor_id', 'name', 'nationality'},
    'races': {'season', 'round', 'race_name', 'circuit_id', 'date'},
    'results': {'season', 'round', 'race_name', 'driver_id', 'constructor_id', 'grid', 'position', 'points', 'laps', 'status'},
    'qualifying': {'season', 'round', 'driver_id', 'constructor_id', 'position'},
    'driver_standings': {'season', 'position', 'points', 'wins', 'driver_id'},
    'constructor_standings': {'season', 'position', 'points', 'wins', 'constructor_id'},
}
raw = {}
for name, file_name in FILES.items():
    path = DATA_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    raw[name] = pd.read_csv(path)
    raw[name].columns = raw[name].columns.str.strip().str.lower()

schema_report = []
for name, df_ in raw.items():
    missing = sorted(required_columns[name] - set(df_.columns))
    schema_report.append({'table': name, 'rows': len(df_), 'columns': len(df_.columns), 'missing_required_columns': ', '.join(missing) if missing else 'None'})
    if missing:
        raise ValueError(f'{name} is missing required columns: {missing}')
display(pd.DataFrame(schema_report))


## 3. Cleaning, Parsing, and Joins

The analytical table is built at race-driver grain. It joins results to races, circuits, drivers, constructors, and qualifying. It also creates race outcomes such as wins, podiums, DNFs, normalized finishing position, and qualifying-to-race deltas.


In [ ]:
def make_race_id(frame: pd.DataFrame) -> pd.Series:
    return frame['season'].astype(int).astype(str) + '_' + frame['round'].astype(int).astype(str).str.zfill(2)

def parse_lap_time_to_seconds(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return np.nan
    try:
        parts = text.split(':')
        if len(parts) == 1:
            return float(parts[0])
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        if len(parts) == 3:
            return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except ValueError:
        return np.nan
    return np.nan

circuits = raw['circuits'].copy()
drivers = raw['drivers'].copy()
constructors = raw['constructors'].copy()
races = raw['races'].copy()
results = raw['results'].copy()
qualifying = raw['qualifying'].copy()
driver_standings = raw['driver_standings'].copy()
constructor_standings = raw['constructor_standings'].copy()

for frame in [races, results, qualifying]:
    frame['season'] = pd.to_numeric(frame['season'], errors='coerce').astype('Int64')
    frame['round'] = pd.to_numeric(frame['round'], errors='coerce').astype('Int64')
    frame['race_id'] = make_race_id(frame)

races['date'] = pd.to_datetime(races['date'], errors='coerce')
for col in ['grid', 'position', 'points', 'laps', 'fastest_lap', 'fastest_lap_rank']:
    if col in results.columns:
        results[col] = pd.to_numeric(results[col], errors='coerce')

qualifying = qualifying.rename(columns={'position': 'quali_position'})
qualifying['quali_position'] = pd.to_numeric(qualifying['quali_position'], errors='coerce')
for col in ['q1', 'q2', 'q3']:
    if col in qualifying.columns:
        qualifying[f'{col}_seconds'] = qualifying[col].apply(parse_lap_time_to_seconds)

constructors_clean = constructors.rename(columns={'name': 'constructor_name_master', 'nationality': 'constructor_nationality'})
drivers_clean = drivers.rename(columns={'nationality': 'driver_nationality'})
drivers_clean['dob'] = pd.to_datetime(drivers_clean['dob'], errors='coerce')
drivers_clean['driver_name_master'] = (drivers_clean['given_name'].fillna('').str.strip() + ' ' + drivers_clean['family_name'].fillna('').str.strip()).str.strip()
race_calendar = races.merge(circuits, on='circuit_id', how='left')

base = (results
    .merge(race_calendar[['race_id', 'season', 'round', 'race_name', 'date', 'circuit_id', 'circuit_name', 'locality', 'country', 'lat', 'lng']], on=['race_id', 'season', 'round', 'race_name'], how='left')
    .merge(drivers_clean[['driver_id', 'driver_name_master', 'dob', 'driver_nationality', 'number', 'code']], on='driver_id', how='left')
    .merge(constructors_clean[['constructor_id', 'constructor_name_master', 'constructor_nationality']], on='constructor_id', how='left')
    .merge(qualifying[['race_id', 'driver_id', 'quali_position', 'q1_seconds', 'q2_seconds', 'q3_seconds']], on=['race_id', 'driver_id'], how='left'))

base['driver_name'] = base['driver_name'].fillna(base['driver_name_master'])
base['constructor_name'] = base['constructor'].fillna(base['constructor_name_master'])
base['grid_clean'] = base['grid'].where(base['grid'] > 0)
base['field_size'] = base.groupby('race_id')['driver_id'].transform('count')
base['position_norm'] = (1 - ((base['position'] - 1) / (base['field_size'] - 1).replace(0, np.nan))).clip(0, 1)
base['podium'] = (base['position'] <= 3).astype(int)
base['win'] = (base['position'] == 1).astype(int)
base['points_finish'] = (base['points'] > 0).astype(int)
base['race_gain_from_grid'] = base['grid_clean'] - base['position']
base['quali_to_race_delta'] = base['position'] - base['quali_position']
base['driver_age'] = ((base['date'] - base['dob']).dt.days / 365.25).where(base['dob'].notna())
finished_status = base['status'].fillna('').str.contains(r'finished|\+\d+ lap', case=False, regex=True)
base['dnf'] = (~finished_status).astype(int)
base['classified_finish'] = finished_status.astype(int)
season_race_counts = races.groupby('season')['race_id'].nunique().rename('season_races')
season_points = base.groupby('season')['points'].sum().rename('season_total_points')
base = base.merge(season_race_counts, on='season', how='left').merge(season_points, on='season', how='left')
base['race_points_share'] = base['points'] / base.groupby('race_id')['points'].transform('sum').replace(0, np.nan)

era_bins = [1949, 1959, 1969, 1979, 1989, 1999, 2009, 2013, 2021, 2026]
era_labels = ['1950s: birth', '1960s: danger and innovation', '1970s: aero emergence', '1980s: turbo era', '1990s: electronics and factories', '2000s: tire wars and V8 transition', '2010-2013: V8 finale', '2014-2021: hybrid turbo', '2022-2026: ground effect']
base['era'] = pd.cut(base['season'], bins=era_bins, labels=era_labels, include_lowest=True)

display(pd.DataFrame({'metric': ['race-result rows', 'races', 'drivers', 'constructors', 'seasons', 'qualifying coverage rows'], 'value': [len(base), base['race_id'].nunique(), base['driver_id'].nunique(), base['constructor_id'].nunique(), f"{int(base['season'].min())}-{int(base['season'].max())}", int(base['quali_position'].notna().sum())]}))
display(base.head())


## 4. Feature Engineering

Advanced features include Driver Performance Index, Constructor Dominance Score, finishing consistency, qualifying versus race deltas, career longevity, and peak performance indicators.


In [ ]:
driver_season = base.groupby(['season', 'driver_id', 'driver_name'], dropna=False).agg(starts=('race_id', 'nunique'), avg_finish=('position', 'mean'), median_finish=('position', 'median'), finish_std=('position', 'std'), avg_position_norm=('position_norm', 'mean'), points=('points', 'sum'), wins=('win', 'sum'), podiums=('podium', 'sum'), dnfs=('dnf', 'sum'), avg_grid=('grid_clean', 'mean'), avg_quali=('quali_position', 'mean'), avg_gain_from_grid=('race_gain_from_grid', 'mean'), avg_quali_to_race_delta=('quali_to_race_delta', 'mean'), avg_field_size=('field_size', 'mean')).reset_index()
driver_season = driver_season.merge(season_race_counts, on='season', how='left').merge(season_points, on='season', how='left')
driver_season['win_rate'] = driver_season['wins'] / driver_season['starts']
driver_season['podium_rate'] = driver_season['podiums'] / driver_season['starts']
driver_season['dnf_rate'] = driver_season['dnfs'] / driver_season['starts']
driver_season['reliability'] = 1 - driver_season['dnf_rate']
driver_season['driver_points_share'] = driver_season['points'] / driver_season['season_total_points'].replace(0, np.nan)
driver_season['quali_score'] = (1 - ((driver_season['avg_quali'] - 1) / (driver_season['avg_field_size'] - 1).replace(0, np.nan))).fillna(driver_season['avg_position_norm']).clip(0, 1)
driver_season['driver_performance_index'] = 100 * (0.35 * driver_season['avg_position_norm'].fillna(0) + 0.25 * driver_season['driver_points_share'].fillna(0) + 0.18 * driver_season['podium_rate'].fillna(0) + 0.12 * driver_season['win_rate'].fillna(0) + 0.10 * driver_season['reliability'].fillna(0))

constructor_season = base.groupby(['season', 'constructor_id', 'constructor_name'], dropna=False).agg(entries=('race_id', 'count'), races_entered=('race_id', 'nunique'), avg_finish=('position', 'mean'), avg_position_norm=('position_norm', 'mean'), points=('points', 'sum'), wins=('win', 'sum'), podiums=('podium', 'sum'), dnfs=('dnf', 'sum'), drivers=('driver_id', 'nunique')).reset_index()
constructor_season = constructor_season.merge(season_race_counts, on='season', how='left').merge(season_points, on='season', how='left')
constructor_season['points_share'] = constructor_season['points'] / constructor_season['season_total_points'].replace(0, np.nan)
constructor_season['win_rate'] = constructor_season['wins'] / constructor_season['season_races'].replace(0, np.nan)
constructor_season['podium_rate'] = constructor_season['podiums'] / (constructor_season['season_races'] * 3).replace(0, np.nan)
constructor_season['constructor_dominance_score'] = 100 * (0.35 * constructor_season['points_share'].fillna(0) + 0.30 * constructor_season['win_rate'].fillna(0) + 0.20 * constructor_season['podium_rate'].fillna(0) + 0.15 * constructor_season['avg_position_norm'].fillna(0))

circuit_profile = base.groupby(['circuit_id', 'circuit_name', 'country'], dropna=False).agg(races=('race_id', 'nunique'), avg_field_size=('field_size', 'mean'), avg_dnf_rate=('dnf', 'mean'), avg_grid_gain=('race_gain_from_grid', 'mean'), unique_winners=('driver_id', lambda s: base.loc[s.index][base.loc[s.index, 'win'].eq(1)]['driver_id'].nunique())).reset_index()
circuit_profile = circuit_profile.merge(base[base['quali_position'].eq(1)].groupby('circuit_id')['win'].mean().rename('pole_win_rate'), on='circuit_id', how='left')
circuit_profile['winner_diversity'] = circuit_profile['unique_winners'] / circuit_profile['races'].replace(0, np.nan)

career = base.groupby(['driver_id', 'driver_name'], dropna=False).agg(starts=('race_id', 'nunique'), first_season=('season', 'min'), last_season=('season', 'max'), seasons=('season', 'nunique'), wins=('win', 'sum'), podiums=('podium', 'sum'), points=('points', 'sum'), dnfs=('dnf', 'sum'), avg_finish=('position', 'mean'), finish_std=('position', 'std'), avg_position_norm=('position_norm', 'mean')).reset_index()
career['win_rate'] = career['wins'] / career['starts'].replace(0, np.nan)
career['podium_rate'] = career['podiums'] / career['starts'].replace(0, np.nan)
career['dnf_rate'] = career['dnfs'] / career['starts'].replace(0, np.nan)
career['career_span'] = career['last_season'] - career['first_season'] + 1
career['finish_consistency'] = 1 / (1 + career['finish_std'].fillna(career['finish_std'].median()))
career = career.merge(driver_season.groupby('driver_id').agg(avg_dpi=('driver_performance_index', 'mean'), peak_dpi=('driver_performance_index', 'max'), top3_dpi=('driver_performance_index', lambda s: s.nlargest(min(3, len(s))).mean())).reset_index(), on='driver_id', how='left')
for col in ['wins', 'podiums', 'win_rate', 'podium_rate', 'avg_position_norm', 'peak_dpi', 'top3_dpi', 'seasons', 'finish_consistency']:
    career[f'{col}_rank'] = career[col].rank(pct=True)
career['greatness_score'] = 100 * (0.16 * career['wins_rank'] + 0.14 * career['podiums_rank'] + 0.14 * career['win_rate_rank'] + 0.12 * career['podium_rate_rank'] + 0.12 * career['avg_position_norm_rank'] + 0.14 * career['peak_dpi_rank'] + 0.10 * career['top3_dpi_rank'] + 0.05 * career['seasons_rank'] + 0.03 * career['finish_consistency_rank'])
career_leaderboard = career.sort_values('greatness_score', ascending=False).reset_index(drop=True)
constructor_leaderboard = constructor_season.sort_values('constructor_dominance_score', ascending=False).reset_index(drop=True)
display(career_leaderboard[['driver_name', 'first_season', 'last_season', 'starts', 'wins', 'podiums', 'win_rate', 'peak_dpi', 'greatness_score']].head(15))
display(constructor_leaderboard[['season', 'constructor_name', 'points', 'wins', 'points_share', 'constructor_dominance_score']].head(15))


In [ ]:
top_driver = career_leaderboard.iloc[0]
top_constructor_season = constructor_leaderboard.iloc[0]
most_reliable = career[career['starts'] >= 50].sort_values('dnf_rate').iloc[0]
display(Markdown(f'''
### Feature Engineering Snapshot

- The highest composite driver score belongs to **{top_driver['driver_name']}** with a greatness score of **{top_driver['greatness_score']:.1f}**.
- The strongest constructor-season by this dominance formula is **{top_constructor_season['constructor_name']} {int(top_constructor_season['season'])}**, scoring **{top_constructor_season['constructor_dominance_score']:.1f}**.
- Among drivers with at least 50 starts, the lowest DNF rate is **{most_reliable['driver_name']}** at **{most_reliable['dnf_rate']:.1%}**.
'''))


## 5. Exploratory Data Analysis

This section visualizes driver performance over time, constructor dominance, circuit characteristics, distributions of wins, podiums, DNFs, and race-level correlations.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
top15 = career_leaderboard.head(15).sort_values('greatness_score')
sns.barplot(data=top15, x='greatness_score', y='driver_name', ax=ax, palette='viridis')
ax.set_title('Statistical Greatness Score: Top 15 Drivers')
ax.set_xlabel('Composite greatness score')
ax.set_ylabel('')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

selected_drivers = career_leaderboard.head(8)['driver_id'].tolist()
plot_df = driver_season[driver_season['driver_id'].isin(selected_drivers)].copy()
fig, ax = plt.subplots(figsize=(13, 7))
sns.lineplot(data=plot_df, x='season', y='driver_performance_index', hue='driver_name', marker='o', linewidth=2, ax=ax)
ax.set_title('Driver Performance Index Over Time: Elite Careers')
ax.set_xlabel('Season')
ax.set_ylabel('Driver Performance Index')
ax.legend(title='Driver', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
top_constructor_ids = constructor_season.groupby('constructor_id')['constructor_dominance_score'].max().sort_values(ascending=False).head(10).index
constructor_plot = constructor_season[constructor_season['constructor_id'].isin(top_constructor_ids)].copy()
fig, ax = plt.subplots(figsize=(13, 7))
sns.lineplot(data=constructor_plot, x='season', y='constructor_dominance_score', hue='constructor_name', linewidth=2, ax=ax)
ax.axhline(70, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_title('Constructor Dominance Across Eras')
ax.set_xlabel('Season')
ax.set_ylabel('Constructor dominance score')
ax.legend(title='Constructor', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
career['wins'].clip(upper=career['wins'].quantile(0.99)).hist(ax=axes[0], bins=30, color='#4c78a8')
axes[0].set_title('Distribution of Career Wins')
axes[0].set_xlabel('Wins, 99th percentile clipped')
axes[0].set_ylabel('Drivers')
career['podiums'].clip(upper=career['podiums'].quantile(0.99)).hist(ax=axes[1], bins=30, color='#f58518')
axes[1].set_title('Distribution of Career Podiums')
axes[1].set_xlabel('Podiums, 99th percentile clipped')
axes[1].set_ylabel('Drivers')
career.loc[career['starts'] >= 10, 'dnf_rate'].hist(ax=axes[2], bins=30, color='#54a24b')
axes[2].set_title('Distribution of DNF Rate')
axes[2].set_xlabel('DNF rate, drivers with 10+ starts')
axes[2].set_ylabel('Drivers')
plt.tight_layout()
plt.show()


In [ ]:
circuit_plot = circuit_profile[circuit_profile['races'] >= 5].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
hardest = circuit_plot.sort_values('avg_dnf_rate', ascending=False).head(12).sort_values('avg_dnf_rate')
sns.barplot(data=hardest, x='avg_dnf_rate', y='circuit_name', ax=axes[0], palette='rocket')
axes[0].set_title('Highest Average DNF Rate by Circuit')
axes[0].set_xlabel('Average DNF rate')
axes[0].set_ylabel('')
pole_sensitive = circuit_plot.dropna(subset=['pole_win_rate']).sort_values('pole_win_rate', ascending=False).head(12).sort_values('pole_win_rate')
sns.barplot(data=pole_sensitive, x='pole_win_rate', y='circuit_name', ax=axes[1], palette='mako')
axes[1].set_title('Highest Pole-to-Win Conversion by Circuit')
axes[1].set_xlabel('Pole win rate')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
correlation_cols = ['season', 'round', 'grid_clean', 'quali_position', 'position', 'points', 'laps', 'field_size', 'position_norm', 'podium', 'win', 'dnf', 'race_gain_from_grid', 'quali_to_race_delta', 'driver_age', 'race_points_share']
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(base[correlation_cols].corr(numeric_only=True), cmap='vlag', center=0, annot=False, linewidths=0.3, ax=ax)
ax.set_title('Correlation Heatmap: Race-Level Features')
plt.tight_layout()
plt.show()

display(Markdown('''
### EDA Reading Notes

- Career wins and podiums are highly skewed, which is typical of elite motorsport.
- Constructor strength appears in waves, often around regulations, engines, tires, and organizational advantage.
- Circuit DNF and pole-conversion rates help separate venue effects from pure team or driver strength.
- Qualifying and grid position are major predictors, but reliability, field spread, and circuit profile mediate the relationship.
'''))


## 6. Advanced Historical Analysis

This section identifies greatest drivers statistically, compares eras, detects constructor dynasties, and measures qualifying importance.


In [ ]:
display(career_leaderboard[['driver_name', 'first_season', 'last_season', 'starts', 'wins', 'podiums', 'win_rate', 'podium_rate', 'dnf_rate', 'peak_dpi', 'top3_dpi', 'greatness_score']].head(25).style.format({'win_rate': '{:.1%}', 'podium_rate': '{:.1%}', 'dnf_rate': '{:.1%}', 'peak_dpi': '{:.1f}', 'top3_dpi': '{:.1f}', 'greatness_score': '{:.1f}'}))

era_summary = base.groupby('era', observed=True).agg(seasons=('season', 'nunique'), races=('race_id', 'nunique'), avg_field_size=('field_size', 'mean'), dnf_rate=('dnf', 'mean'), avg_points_finish_rate=('points_finish', 'mean'), avg_grid_gain_abs=('race_gain_from_grid', lambda s: s.abs().mean()), quali_available_rate=('quali_position', lambda s: s.notna().mean())).reset_index()
display(era_summary.style.format({'avg_field_size': '{:.1f}', 'dnf_rate': '{:.1%}', 'avg_points_finish_rate': '{:.1%}', 'avg_grid_gain_abs': '{:.2f}', 'quali_available_rate': '{:.1%}'}))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=era_summary, x='dnf_rate', y='era', ax=axes[0], palette='flare')
axes[0].set_title('Reliability by Era')
axes[0].set_xlabel('DNF rate')
axes[0].set_ylabel('')
sns.barplot(data=era_summary, x='avg_grid_gain_abs', y='era', ax=axes[1], palette='crest')
axes[1].set_title('Average Absolute Grid-to-Finish Movement by Era')
axes[1].set_xlabel('Mean absolute position change')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()


In [ ]:
def find_dynasties(data, threshold=55, min_length=2):
    rows = []
    for constructor_id, group in data.sort_values(['constructor_id', 'season']).groupby('constructor_id'):
        group = group[group['constructor_dominance_score'] >= threshold].sort_values('season')
        run, previous = [], None
        for _, row in group.iterrows():
            if previous is None or int(row['season']) == previous + 1:
                run.append(row)
            else:
                if len(run) >= min_length:
                    run_df = pd.DataFrame(run)
                    rows.append({'constructor_id': constructor_id, 'constructor_name': run_df['constructor_name'].iloc[0], 'start_season': int(run_df['season'].min()), 'end_season': int(run_df['season'].max()), 'length': len(run_df), 'avg_dominance_score': run_df['constructor_dominance_score'].mean(), 'total_wins': int(run_df['wins'].sum()), 'total_points': run_df['points'].sum()})
                run = [row]
            previous = int(row['season'])
        if len(run) >= min_length:
            run_df = pd.DataFrame(run)
            rows.append({'constructor_id': constructor_id, 'constructor_name': run_df['constructor_name'].iloc[0], 'start_season': int(run_df['season'].min()), 'end_season': int(run_df['season'].max()), 'length': len(run_df), 'avg_dominance_score': run_df['constructor_dominance_score'].mean(), 'total_wins': int(run_df['wins'].sum()), 'total_points': run_df['points'].sum()})
    return pd.DataFrame(rows).sort_values(['length', 'avg_dominance_score'], ascending=False)

dynasties = find_dynasties(constructor_season)
display(dynasties.head(20).style.format({'avg_dominance_score': '{:.1f}', 'total_points': '{:.1f}'}))

qual_df = base.dropna(subset=['quali_position', 'position']).copy()
season_qual_corr = qual_df.groupby('season').apply(lambda g: g['quali_position'].corr(g['position']) if len(g) >= 10 else np.nan).rename('qualifying_finish_correlation').reset_index()
era_corr = qual_df.groupby('era', observed=True).apply(lambda g: g['quali_position'].corr(g['position'])).rename('avg_quali_finish_corr')
era_qual = qual_df.groupby('era', observed=True).agg(rows=('race_id', 'count'), avg_quali_to_race_delta=('quali_to_race_delta', 'mean'), pole_win_rate=('win', lambda s: qual_df.loc[s.index][qual_df.loc[s.index, 'quali_position'].eq(1)]['win'].mean())).reset_index().merge(era_corr, on='era', how='left')
fig, ax = plt.subplots(figsize=(13, 6))
sns.lineplot(data=season_qual_corr, x='season', y='qualifying_finish_correlation', marker='o', ax=ax)
ax.set_title('Qualifying-to-Finish Correlation by Season')
ax.set_xlabel('Season')
ax.set_ylabel('Correlation: qualifying position vs finish position')
ax.axhline(season_qual_corr['qualifying_finish_correlation'].median(), color='black', linestyle='--', linewidth=1, alpha=0.5)
plt.tight_layout()
plt.show()
display(era_qual.style.format({'avg_quali_finish_corr': '{:.2f}', 'avg_quali_to_race_delta': '{:.2f}', 'pole_win_rate': '{:.1%}'}))


In [ ]:
best_dynasty = dynasties.iloc[0] if not dynasties.empty else None
best_qual_season = season_qual_corr.dropna().sort_values('qualifying_finish_correlation', ascending=False).head(1)
lines = ['### Advanced Analysis Highlights', '', f"- The statistical leaderboard is led by **{career_leaderboard.iloc[0]['driver_name']}**, but rankings are sensitive to how peaks and longevity are weighted."]
if best_dynasty is not None:
    lines.append(f"- The strongest sustained constructor run detected here is **{best_dynasty['constructor_name']} {int(best_dynasty['start_season'])}-{int(best_dynasty['end_season'])}**, averaging **{best_dynasty['avg_dominance_score']:.1f}** dominance points.")
if not best_qual_season.empty:
    lines.append(f"- The highest qualifying-to-finish correlation season in the qualifying era is **{int(best_qual_season.iloc[0]['season'])}** at **{best_qual_season.iloc[0]['qualifying_finish_correlation']:.2f}**.")
lines.append('- Higher qualifying correlation means starting position was more predictive, but strategy, reliability, weather, safety cars, and penalties also shape race results.')
display(Markdown('\n'.join(lines)))


## 7. Machine Learning: Finish Position and Podium Prediction

Two supervised tasks are modeled: finishing position regression and podium classification. Features are pre-race oriented, including calendar context, grid or qualifying information, driver age, and shifted historical form for drivers, constructors, and circuits.


In [ ]:
model_data = base.sort_values(['date', 'season', 'round', 'position']).copy()
model_data['date_ord'] = model_data['date'].map(pd.Timestamp.toordinal)
for key, prefix in [('driver_id', 'driver'), ('constructor_id', 'constructor'), ('circuit_id', 'circuit')]:
    model_data[f'{prefix}_prior_starts'] = model_data.groupby(key).cumcount()
model_data['driver_prior_avg_finish'] = model_data.groupby('driver_id')['position'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['driver_prior_podium_rate'] = model_data.groupby('driver_id')['podium'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['driver_prior_dnf_rate'] = model_data.groupby('driver_id')['dnf'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['driver_prior_avg_gain'] = model_data.groupby('driver_id')['race_gain_from_grid'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['constructor_prior_avg_finish'] = model_data.groupby('constructor_id')['position'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['constructor_prior_win_rate'] = model_data.groupby('constructor_id')['win'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['constructor_prior_podium_rate'] = model_data.groupby('constructor_id')['podium'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['circuit_prior_dnf_rate'] = model_data.groupby('circuit_id')['dnf'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['circuit_prior_pole_win_rate'] = model_data.assign(pole_win=lambda d: ((d['quali_position'] == 1) & (d['win'] == 1)).astype(int)).groupby('circuit_id')['pole_win'].expanding().mean().shift().reset_index(level=0, drop=True)
model_data['grid_or_quali'] = model_data['grid_clean'].fillna(model_data['quali_position'])
model_data['has_qualifying'] = model_data['quali_position'].notna().astype(int)
model_data['target_position'] = model_data['position'].astype(float)
model_data['target_podium'] = model_data['podium'].astype(int)

numeric_features = ['season', 'round', 'field_size', 'driver_age', 'grid_clean', 'quali_position', 'grid_or_quali', 'has_qualifying', 'driver_prior_starts', 'driver_prior_avg_finish', 'driver_prior_podium_rate', 'driver_prior_dnf_rate', 'driver_prior_avg_gain', 'constructor_prior_starts', 'constructor_prior_avg_finish', 'constructor_prior_win_rate', 'constructor_prior_podium_rate', 'circuit_prior_starts', 'circuit_prior_dnf_rate', 'circuit_prior_pole_win_rate']
categorical_features = ['driver_id', 'constructor_id', 'circuit_id', 'era']
ml_df = model_data.dropna(subset=['target_position', 'target_podium', 'date']).copy()
unique_seasons = sorted(ml_df['season'].dropna().unique())
season_cutoff = unique_seasons[max(1, int(len(unique_seasons) * 0.80))]
train_df = ml_df[ml_df['season'] < season_cutoff].copy()
test_df = ml_df[ml_df['season'] >= season_cutoff].copy()
if len(test_df) < 500:
    train_df, test_df = train_test_split(ml_df, test_size=0.2, random_state=RANDOM_STATE, stratify=ml_df['target_podium'])
    split_description = 'random stratified 80/20 split'
else:
    split_description = f'temporal split: train < {season_cutoff}, test >= {season_cutoff}'
X_train = train_df[numeric_features + categorical_features]
X_test = test_df[numeric_features + categorical_features]
y_train_reg = train_df['target_position']
y_test_reg = test_df['target_position']
y_train_clf = train_df['target_podium']
y_test_clf = test_df['target_podium']
try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=False)
preprocess = ColumnTransformer(transformers=[('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', onehot)]), categorical_features)])
print(split_description)
print(f'Train rows: {len(train_df):,} | Test rows: {len(test_df):,}')
print(f'Podium rate in train: {y_train_clf.mean():.1%} | test: {y_test_clf.mean():.1%}')


In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))
regression_models = {'Linear Regression': LinearRegression(), 'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)}
if HAS_XGBOOST:
    regression_models['XGBoost'] = XGBRegressor(n_estimators=450, max_depth=5, learning_rate=0.035, subsample=0.85, colsample_bytree=0.85, objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=-1)
regression_results, regression_pipelines = [], {}
for name, model in regression_models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train_reg)
    preds = pipe.predict(X_test)
    regression_results.append({'model': name, 'RMSE': rmse(y_test_reg, preds), 'MAE': mean_absolute_error(y_test_reg, preds)})
    regression_pipelines[name] = pipe
regression_results_df = pd.DataFrame(regression_results).sort_values('RMSE')
display(regression_results_df.style.format({'RMSE': '{:.3f}', 'MAE': '{:.3f}'}))
best_regression_model_name = regression_results_df.iloc[0]['model']
best_regression_pipeline = regression_pipelines[best_regression_model_name]
print(f'Best finishing-position model: {best_regression_model_name}')

classification_models = {'Logistic Regression': LogisticRegression(max_iter=1500, class_weight='balanced', random_state=RANDOM_STATE), 'Random Forest': RandomForestClassifier(n_estimators=350, min_samples_leaf=3, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)}
if HAS_XGBOOST:
    scale_pos_weight = (len(y_train_clf) - y_train_clf.sum()) / max(y_train_clf.sum(), 1)
    classification_models['XGBoost'] = XGBClassifier(n_estimators=450, max_depth=4, learning_rate=0.035, subsample=0.85, colsample_bytree=0.85, objective='binary:logistic', eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=-1)
classification_results, classification_pipelines = [], {}
for name, model in classification_models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train_clf)
    pred_labels = pipe.predict(X_test)
    pred_probs = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else pred_labels
    try:
        auc = roc_auc_score(y_test_clf, pred_probs)
    except ValueError:
        auc = np.nan
    classification_results.append({'model': name, 'Accuracy': accuracy_score(y_test_clf, pred_labels), 'F1': f1_score(y_test_clf, pred_labels, zero_division=0), 'ROC_AUC': auc})
    classification_pipelines[name] = pipe
classification_results_df = pd.DataFrame(classification_results).sort_values('F1', ascending=False)
display(classification_results_df.style.format({'Accuracy': '{:.3f}', 'F1': '{:.3f}', 'ROC_AUC': '{:.3f}'}))
best_classification_model_name = classification_results_df.iloc[0]['model']
best_classification_pipeline = classification_pipelines[best_classification_model_name]
print(f'Best podium model by F1: {best_classification_model_name}')


In [ ]:
best_clf_preds = best_classification_pipeline.predict(X_test)
cm = confusion_matrix(y_test_clf, best_clf_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title(f'Podium Prediction Confusion Matrix: {best_classification_model_name}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticklabels(['No Podium', 'Podium'])
ax.set_yticklabels(['No Podium', 'Podium'], rotation=0)
plt.tight_layout()
plt.show()

ml_summary = f'''
### Modeling Takeaways

- The notebook uses a **{split_description}** to approximate future-race prediction.
- The best finishing-position model is **{best_regression_model_name}** with RMSE **{regression_results_df.iloc[0]['RMSE']:.2f}** positions.
- The best podium classifier by F1 is **{best_classification_model_name}** with F1 **{classification_results_df.iloc[0]['F1']:.2f}**.
- Grid and qualifying features are powerful, but rolling driver and constructor form add useful historical context.
'''
if not HAS_XGBOOST:
    ml_summary += '\n- XGBoost is not installed in this runtime, so XGBoost definitions were skipped while the notebook remained executable.'
display(Markdown(ml_summary))


## 8. Driver Clustering with PCA and KMeans

Drivers with at least 10 starts are clustered into four archetypes using career features. PCA provides a two-dimensional visualization of the clustering space.


In [ ]:
cluster_df = career[career['starts'] >= 10].copy()
cluster_features = ['starts', 'seasons', 'wins', 'podiums', 'win_rate', 'podium_rate', 'dnf_rate', 'avg_finish', 'avg_position_norm', 'finish_consistency', 'avg_dpi', 'peak_dpi', 'top3_dpi', 'greatness_score']
cluster_matrix = cluster_df[cluster_features].replace([np.inf, -np.inf], np.nan)
cluster_matrix = cluster_matrix.fillna(cluster_matrix.median(numeric_only=True))
X_cluster = StandardScaler().fit_transform(cluster_matrix)
kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=20)
cluster_df['cluster'] = kmeans.fit_predict(X_cluster)
cluster_order = cluster_df.groupby('cluster')['greatness_score'].mean().sort_values(ascending=False).index.tolist()
cluster_names = {cluster_order[0]: 'Elite / title-impact', cluster_order[1]: 'High-quality consistent', cluster_order[2]: 'Midfield / points-capable', cluster_order[3]: 'Limited-impact / short-career'}
cluster_df['driver_category'] = cluster_df['cluster'].map(cluster_names)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_components = pca.fit_transform(X_cluster)
cluster_df['pca_1'] = pca_components[:, 0]
cluster_df['pca_2'] = pca_components[:, 1]
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(data=cluster_df, x='pca_1', y='pca_2', hue='driver_category', size='starts', sizes=(30, 260), alpha=0.82, ax=ax)
ax.set_title('Driver Archetypes: PCA Projection of Career Feature Clusters')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Cluster')
plt.tight_layout()
plt.show()
cluster_profile = cluster_df.groupby('driver_category').agg(drivers=('driver_id', 'count'), avg_starts=('starts', 'mean'), avg_wins=('wins', 'mean'), avg_podiums=('podiums', 'mean'), avg_win_rate=('win_rate', 'mean'), avg_podium_rate=('podium_rate', 'mean'), avg_greatness_score=('greatness_score', 'mean')).sort_values('avg_greatness_score', ascending=False)
display(cluster_profile.style.format({'avg_starts': '{:.1f}', 'avg_wins': '{:.1f}', 'avg_podiums': '{:.1f}', 'avg_win_rate': '{:.1%}', 'avg_podium_rate': '{:.1%}', 'avg_greatness_score': '{:.1f}'}))


## 9. Race Outcome Simulation Engine

The simulation engine takes the latest model-ready race, predicts expected finishing positions, and runs Monte Carlo simulations around those expectations. Uncertainty is scaled by driver historical DNF rate.


In [ ]:
def simulate_race_outcome(race_frame, regression_pipeline, n_simulations=5000, noise_scale=2.75, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    sim_frame = race_frame.copy().reset_index(drop=True)
    X_race = sim_frame[numeric_features + categorical_features]
    expected_position = regression_pipeline.predict(X_race)
    sim_frame['expected_position'] = expected_position
    driver_noise = sim_frame['driver_prior_dnf_rate'].fillna(sim_frame['driver_prior_dnf_rate'].median()).fillna(0.20).to_numpy()
    uncertainty = noise_scale * (1 + driver_noise)
    win_counts = np.zeros(len(sim_frame), dtype=int)
    podium_counts = np.zeros(len(sim_frame), dtype=int)
    avg_rank_accumulator = np.zeros(len(sim_frame), dtype=float)
    for _ in range(n_simulations):
        sampled_strength = expected_position + rng.normal(0, uncertainty, size=len(sim_frame))
        order = np.argsort(sampled_strength)
        ranks = np.empty(len(sim_frame), dtype=int)
        ranks[order] = np.arange(1, len(sim_frame) + 1)
        win_counts += (ranks == 1)
        podium_counts += (ranks <= 3)
        avg_rank_accumulator += ranks
    sim_frame['sim_win_probability'] = win_counts / n_simulations
    sim_frame['sim_podium_probability'] = podium_counts / n_simulations
    sim_frame['sim_avg_finish'] = avg_rank_accumulator / n_simulations
    return sim_frame.sort_values('sim_avg_finish')

latest_model_race_id = ml_df.dropna(subset=['grid_or_quali']).sort_values('date').iloc[-1]['race_id']
latest_race_frame = ml_df[ml_df['race_id'] == latest_model_race_id].copy()
latest_race_label = latest_race_frame[['season', 'round', 'race_name']].drop_duplicates().iloc[0]
simulation_results = simulate_race_outcome(latest_race_frame, best_regression_pipeline, n_simulations=3000)
simulation_table = simulation_results[['driver_name', 'constructor_name', 'grid_clean', 'quali_position', 'position', 'expected_position', 'sim_avg_finish', 'sim_win_probability', 'sim_podium_probability']].head(15)
display(Markdown(f"### Simulation Example: {int(latest_race_label['season'])} {latest_race_label['race_name']}"))
display(simulation_table.style.format({'grid_clean': '{:.0f}', 'quali_position': '{:.0f}', 'position': '{:.0f}', 'expected_position': '{:.2f}', 'sim_avg_finish': '{:.2f}', 'sim_win_probability': '{:.1%}', 'sim_podium_probability': '{:.1%}'}))
fig, ax = plt.subplots(figsize=(12, 7))
plot_sim = simulation_results.head(12).sort_values('sim_win_probability')
sns.barplot(data=plot_sim, x='sim_win_probability', y='driver_name', hue='constructor_name', dodge=False, ax=ax)
ax.set_title(f"Monte Carlo Win Probability: {int(latest_race_label['season'])} {latest_race_label['race_name']}")
ax.set_xlabel('Simulated win probability')
ax.set_ylabel('')
ax.legend(title='Constructor', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
sim_top = simulation_results.iloc[0]
display(Markdown(f'''
### Simulation Reading Notes

- The simulation favorite for the selected race is **{sim_top['driver_name']}**, with estimated win probability of **{sim_top['sim_win_probability']:.1%}**.
- The engine converts model-predicted finishing positions into probabilistic outcomes with uncertainty.
- Production improvements would include calibrated uncertainty, weather, tire strategy, safety-car likelihood, penalties, and session-specific pace indicators.
'''))


## 10. Conclusions, Limitations, and Future Work

### Key Insights

- Dominance is cyclical. Constructor strength appears in eras and dynasties, often aligned with regulation changes and technical advantages.
- Greatness has multiple shapes. Peak performance, longevity, reliability, and win conversion tell different stories.
- Qualifying matters, but context matters more. Starting position is a major predictor, yet circuit profile, reliability, and team strength modify its impact.
- Race prediction is feasible but uncertainty-heavy. Models can learn strong patterns from grid, form, team strength, and circuit history, but motorsport outcomes include strategy, incidents, mechanical reliability, weather, and penalties.
- Clustering adds interpretive value. PCA and KMeans separate drivers into recognizable career archetypes beyond simple rank ordering.

### Limitations

- Points systems changed repeatedly, so raw points are not directly comparable across eras.
- Qualifying data starts in 1994 in this archive, limiting comparisons with earlier decades.
- The dataset does not include lap-by-lap pace, tire compounds, pit stops, weather, safety cars, penalties, team orders, or mechanical component data.
- Engineered scores compress complex history into interpretable numbers; they should support discussion, not end it.

### Future Improvements

- Add lap time, pit stop, tire, weather, and safety car data.
- Build era-specific models for changing race formats and reliability regimes.
- Calibrate simulation probabilities through historical backtesting.
- Add Bayesian driver and constructor rating systems.
- Use SHAP or permutation importance for deeper model interpretation.
- Create an interactive dashboard with filters for era, constructor, circuit, and driver nationality.


In [ ]:
final_summary = pd.DataFrame({'artifact': ['Race-result rows analyzed', 'Unique races', 'Unique drivers', 'Unique constructors', 'Seasons covered', 'Best regression model', 'Best podium model', 'Driver clusters'], 'value': [f'{len(base):,}', f'{base["race_id"].nunique():,}', f'{base["driver_id"].nunique():,}', f'{base["constructor_id"].nunique():,}', f"{int(base['season'].min())}-{int(base['season'].max())}", best_regression_model_name, best_classification_model_name, cluster_df['driver_category'].nunique()]})
display(final_summary)
